# OCR Model Selection Lab — Colab CPU/GPU

Ce notebook utilise le même moteur que l’application locale et Docker.

1. Dans **Exécution > Modifier le type d’exécution**, choisissez `CPU` ou `T4 GPU`.
2. Exécutez les cellules dans l’ordre.
3. Le GPU accélère les adaptateurs compatibles, notamment EasyOCR.
4. Un timeout arrête la mesure officielle, mais la réponse tardive et le champ `thinking/reasoning` restent enregistrés dans `traces.jsonl`.
5. Les tokens/s ne sont pertinents que pour les modèles génératifs qui exposent leurs compteurs.

In [ ]:
# Source du projet. Le dépôt étant privé, laissez vide et chargez un ZIP,
# ou saisissez un Personal Access Token GitHub en secret dans la cellule suivante.
REPO_URL = "https://github.com/Eric-Kambire/ocr-model-selection-lab.git"
USE_PRIVATE_GITHUB_REPO = False
BRANCH = "main"
PROJECT_DIR = "/content/ocr-model-selection-lab"


In [ ]:
import base64, getpass, os, pathlib, shutil, subprocess, zipfile

project = pathlib.Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)

if USE_PRIVATE_GITHUB_REPO:
    token = getpass.getpass("GitHub PAT (lecture du dépôt privé) : ")
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}"})
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True, env=git_env)
    finally:
        del token, credential, git_env
else:
    from google.colab import files
    print("Chargez le ZIP du dépôt OCR Model Selection Lab.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if not zip_name:
        raise ValueError("Aucun fichier ZIP reçu.")
    project.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(project)
    candidates = [p.parent for p in project.rglob("main.py")]
    if len(candidates) != 1:
        raise RuntimeError(f"Impossible d’identifier la racine : {candidates}")
    project = candidates[0]

os.chdir(project)
print("Projet chargé depuis :", project)

In [ ]:
!python -m pip install -q -r requirements-ocr.txt


In [ ]:
import platform, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python :", platform.python_version())
print("PyTorch :", torch.__version__)
print("Matériel :", DEVICE)
if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA :", torch.version.cuda)
else:
    print("CPU actif. Pour tester le GPU : Exécution > Modifier le type d’exécution > T4 GPU.")

## Installer et télécharger les modèles

Trois familles sont prévues :

- `mock:*` : modèle simulé, rapide, sert à vérifier que le benchmark fonctionne.
- `easyocr:EasyOCR-Local` : moteur OCR classique recommandé pour Colab. Il utilise le GPU automatiquement si disponible.
- futurs adaptateurs Python/API : meilleur chemin pour brancher des modèles Hugging Face, cloud ou propriétaires.
- `ollama:<nom-du-modèle>` : option avancée seulement si vous voulez lancer un serveur Ollama dans Colab. Ce n’est pas nécessaire pour utiliser Colab.

Conseil pratique : commencez avec `mock`, puis `easyocr`. N’activez Ollama que si vous savez explicitement que le modèle à tester doit tourner via Ollama.

In [ ]:
# Choisissez ici les moteurs à préparer dans Colab.
# Chemin recommandé : EasyOCR ou adaptateurs Python/API.
# EasyOCR télécharge automatiquement ses poids au premier lancement.
# Ollama est optionnel : gardez False sauf besoin explicite d’un serveur Ollama dans Colab.
INSTALL_EASYOCR = True
INSTALL_OLLAMA = False

# Option avancée Ollama seulement.
# Exemples de modèles vision Ollama :
# - "llama3.2-vision" : bon modèle vision, plus lourd.
# - "moondream" : plus léger, utile pour un premier test.
# Changez cette liste selon les modèles que vous voulez benchmarker.
OLLAMA_MODELS_TO_PULL = ["moondream"]

import os, subprocess, textwrap, time

if INSTALL_EASYOCR:
    print("EasyOCR est disponible via requirements-ocr.txt. Les poids seront téléchargés au premier benchmark.")

if INSTALL_OLLAMA:
    print("Installation d’Ollama dans Colab…")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    print("Démarrage du serveur Ollama en arrière-plan…")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
    )
    time.sleep(8)

    for model_name in OLLAMA_MODELS_TO_PULL:
        print(f"Téléchargement du modèle Ollama : {model_name}")
        subprocess.run(["ollama", "pull", model_name], check=True)

    print("Modèles Ollama installés :")
    subprocess.run(["ollama", "list"], check=True)
else:
    print("Ollama ignoré. C’est normal sur Colab sauf si vous voulez explicitement tester un modèle via Ollama.")

## Configurer le benchmark

- `Tout le dataset` utilise tous les documents.
- `Quantité globale` prélève `GLOBAL_QUANTITY` documents.
- `Par catégorie` utilise les limites définies dans `CATEGORY_QUANTITIES`.
- `TIMEOUT_SECONDS` est la durée maximale officielle par image. Une sortie arrivée plus tard est conservée dans la trace, mais n’est pas notée.
- `MODEL_PROMPT` est envoyé aux modèles génératifs compatibles. EasyOCR et les modèles simulés l’ignorent.

In [ ]:
# Modèles à tester. Gardez le préfixe fournisseur : mock:, easyocr:, ollama:, etc.
# Exemples :
# MODEL_SPECS = ["mock:MockOCR-Colab"]
# MODEL_SPECS = ["easyocr:EasyOCR-Local"]
# Option avancée si vous avez activé INSTALL_OLLAMA=True :
# MODEL_SPECS = ["ollama:moondream"]
MODEL_SPECS = ["mock:MockOCR-Colab"]

MODEL_PROMPT = """You are a professional layout-preserving OCR engine.
Transcribe all visible text in the image.
Return only the transcription, without explanation.
Preserve tables and line breaks when possible."""

SELECTION_MODE = "Quantité globale"  # Tout le dataset | Quantité globale | Par catégorie
GLOBAL_QUANTITY = 10
CATEGORY_QUANTITIES = {"bank": 3, "tables": 3, "handwritten": 4}
SHUFFLE = True
SEED = 42
TIMEOUT_SECONDS = 300
MAX_ERRORS = 0  # 0 = aucune limite
EVAL_MODE = "Standard"  # Standard | Bankmark


In [ ]:
import pandas as pd
from main import RUNS_DIR, load_dataset, select_dataset_items
from ocr_benchmark.domain import BenchmarkCase
from ocr_benchmark.registry import build_default_registry
from ocr_benchmark.reporting import RunCheckpoint
from ocr_benchmark.runner import BenchmarkRunner, summarize_results

dataset = load_dataset()
counts = pd.Series([item["category"] for item in dataset]).value_counts()
category_rows = [
    [category, int(available), int(CATEGORY_QUANTITIES.get(category, 0))]
    for category, available in counts.items()
]
selected = select_dataset_items(
    dataset,
    SELECTION_MODE,
    GLOBAL_QUANTITY,
    category_rows,
    shuffle=SHUFFLE,
    seed=SEED,
)
print(f"{len(selected)} document(s), {len(MODEL_SPECS)} modèle(s), {len(selected) * len(MODEL_SPECS)} évaluation(s).")
display(pd.Series([item["category"] for item in selected]).value_counts().rename("documents").to_frame())

In [ ]:
runner = BenchmarkRunner(build_default_registry())
cases = [BenchmarkCase.from_dict(item) for item in selected]

def save_trace(event):
    RunCheckpoint(event["run_id"], RUNS_DIR).append_trace(event)

run_id, results = runner.run(
    MODEL_SPECS,
    cases,
    eval_mode=EVAL_MODE,
    timeout_seconds=TIMEOUT_SECONDS,
    max_errors=MAX_ERRORS,
    model_prompt=MODEL_PROMPT,
    trace=save_trace,
)
summary = summarize_results(results)
run_dir = RunCheckpoint(run_id, RUNS_DIR).finalize(summary, results)
print("Run ID :", run_id)
print("Résultats :", run_dir)
display(summary)

## Examiner les sorties et traces

`status=timeout` signifie que la durée maximale a été dépassée. Une ligne `timing=late_after_timeout` permet d’examiner la réponse reçue ensuite. Les champs bruts sont conservés pour audit avant nettoyage.

In [ ]:
import json

trace_path = run_dir / "traces.jsonl"
traces = []
if trace_path.exists():
    traces = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line]
trace_table = pd.DataFrame(traces)
columns = [name for name in ["model", "image_path", "timing", "status", "latency", "text", "reasoning", "error"] if name in trace_table]
display(trace_table[columns] if columns else trace_table)


In [ ]:
# Télécharger l’ensemble du run : CSV, JSON, rapport et traces brutes.
from google.colab import files
archive = shutil.make_archive(f"/content/{run_id}", "zip", run_dir)
files.download(archive)

## Interface complète

L’interface contient la sélection des quantités, la progression, l’image courante, les métriques en direct et les paramètres. Colab fournit une URL publique temporaire : ne l’utilisez pas avec des documents sensibles.

In [ ]:
from main import build_ui

app = build_ui()
app.launch(share=True, debug=False)